In [ ]:
# =========================
# nb_narrative_generation
# =========================
# Phase 9, SC-16: Generate human-readable narrative summaries per DFM
# Summarizes holdings, key findings, data quality, and recommendations

# ---------- Parameters ----------
period = "2026-03"
run_id = "manual_test_run"

# ---------- Imports ----------
from pyspark.sql import functions as F
import sys
import json

# ---------- Pre-flight checks ----------
try:
    mssparkutils
except NameError:
    print("[ERROR] mssparkutils not available - not running in Fabric context")
    sys.exit(1)

try:
    if not period or not period.strip():
        raise ValueError("period parameter is empty")
    if not run_id or not run_id.strip():
        raise ValueError("run_id parameter is empty")
    print(f"[nb_narrative_generation] START period={period}, run_id={run_id}")
except ValueError as e:
    print(f"[ERROR] Invalid parameters: {e}")
    sys.exit(1)

if not spark.catalog.tableExists("canonical_holdings") or not spark.catalog.tableExists("ai_narrative_summary"):
    print("[ERROR] Required tables do not exist")
    sys.exit(1)

# Load shared_ai_utils
sys.path.append('/lakehouse/default/Files/config')
try:
    from shared_ai_utils import load_ai_config
    ai_enabled = True
except ImportError as e:
    print(f"[WARNING] shared_ai_utils not found: {e}")
    print("[nb_narrative_generation] AI not available; skipping")
    mssparkutils.notebook.exit("AI_DISABLED")

# Load AI config
try:
    ai_client = load_ai_config('/lakehouse/default/Files/config/azure_openai_config.json')
    if ai_client is None:
        raise RuntimeError("AI config returned None")
except Exception as e:
    print(f"[ERROR] Failed to load AI config: {type(e).__name__}: {e}")
    mssparkutils.notebook.exit("CONFIG_ERROR")

# ---------- Load data per DFM ----------
try:
    ch = spark.table("canonical_holdings").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id)
    )
    
    if ch.rdd.isEmpty():
        print(f"[nb_narrative_generation] No canonical_holdings for period={period}, run_id={run_id}")
        mssparkutils.notebook.exit("NO_DATA")
    
    dfms = [r["dfm_id"] for r in ch.select("dfm_id").distinct().collect()]
    print(f"[nb_narrative_generation] Found {len(dfms)} DFMs to summarize")
    
    # Get aggregated stats
    pa = spark.table("policy_aggregates").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id)
    )
    ve = spark.table("validation_events").filter(
        (F.col("period") == period) & (F.col("run_id") == run_id)
    )

except Exception as e:
    print(f"[ERROR] Failed to load data: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)

# ---------- Generate narratives ----------
narratives = []
success_count = 0
error_count = 0

try:
    for dfm_id in dfms:
        try:
            dfm_id_str = str(dfm_id) if dfm_id else "UNKNOWN"
            dfm_ch = ch.filter(F.col("dfm_id") == dfm_id)
            dfm_pa = pa.filter(F.col("dfm_id") == dfm_id)
            dfm_ve = ve.filter(F.col("dfm_id") == dfm_id)
            
            total_holdings = dfm_ch.count()
            total_policies = dfm_pa.count()
            total_value = float(dfm_pa.agg(F.sum("total_bid_value_gbp")).collect()[0][0] or 0)
            
            exceptions = dfm_ve.filter(F.col("status") == "fail").count()
            exception_types = dfm_ve.filter(F.col("status") == "fail").groupBy("rule_id").count().collect()
            
            key_findings = [f"{cnt['rule_id']}: {cnt['count']} exceptions" for cnt in exception_types]
            
            prompt = f"""
Generate a brief executive summary for investment operations:
- DFM: {dfm_id_str}
- Period: {period}
- Total policies: {total_policies}
- Total holdings: {total_holdings}
- Portfolio value (GBP): {total_value:.2f}
- Validation exceptions: {exceptions}
- Exception types: {', '.join(key_findings) if key_findings else 'None'}

Provide:
1. Data quality assessment (brief)
2. Top 3 findings/observations
3. Recommended next actions (max 3)

Return JSON: {{
  "narrative": "...",
  "quality_assessment": "...",
  "findings": [...],
  "recommendations": [...]
}}
"""
            
            try:
                result_text = ai_client.generate_narrative(prompt)
                
                if not result_text or not isinstance(result_text, str):
                    raise ValueError("AI returned non-string response")
                
                result = json.loads(result_text)
                
                narratives.append({
                    "period": period,
                    "run_id": run_id,
                    "dfm_id": dfm_id_str,
                    "narrative_text": str(result.get("narrative", "")),
                    "data_quality_assessment": str(result.get("quality_assessment", "")),
                    "key_findings": json.dumps(result.get("findings", [])),
                    "recommendations": json.dumps(result.get("recommendations", [])),
                    "generated_at": None
                })
                success_count += 1
            
            except json.JSONDecodeError as e:
                error_count += 1
                print(f"[WARNING] AI response not valid JSON for DFM {dfm_id_str}: {e}")
                narratives.append({
                    "period": period,
                    "run_id": run_id,
                    "dfm_id": dfm_id_str,
                    "narrative_text": f"Period {period}: {total_policies} policies, {total_holdings} holdings, GBP {total_value:,.2f}",
                    "data_quality_assessment": "Unable to assess (AI JSON parse error)",
                    "key_findings": json.dumps(key_findings or []),
                    "recommendations": json.dumps(["Manual review recommended"]),
                    "generated_at": None
                })
            except Exception as e:
                error_count += 1
                print(f"[WARNING] Error generating narrative for {dfm_id_str}: {type(e).__name__}: {e}")
                narratives.append({
                    "period": period,
                    "run_id": run_id,
                    "dfm_id": dfm_id_str,
                    "narrative_text": f"Period {period}: {total_policies} policies, {total_holdings} holdings, GBP {total_value:,.2f}",
                    "data_quality_assessment": "Unable to assess (AI unavailable)",
                    "key_findings": json.dumps(key_findings or []),
                    "recommendations": json.dumps(["Manual review recommended"]),
                    "generated_at": None
                })
        
        except Exception as e:
            error_count += 1
            print(f"[WARNING] Failed to process DFM {dfm_id}: {type(e).__name__}: {e}")

    # Write results
    if narratives:
        try:
            df_narratives = spark.createDataFrame(narratives)
            df_narratives.write.mode("append").saveAsTable("ai_narrative_summary")
            print(f"[nb_narrative_generation] COMPLETE generated {len(narratives)} narratives (success={success_count}, errors={error_count})")
        except Exception as e:
            print(f"[ERROR] Failed to write narratives: {type(e).__name__}: {e}")
            sys.exit(1)
    else:
        print("[WARNING] No narratives generated")
    
    mssparkutils.notebook.exit("OK")

except Exception as e:
    print(f"[ERROR] Narrative generation failed: {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()
    sys.exit(1)